In [ ]:
cd ..

In [ ]:
import torch
from src.gaussian_process import LinearGaussianProcess
from src.utils.misc import random_fourier_features
import matplotlib.pyplot as plt
import visualizations.plot_settings as plot_settings

In [3]:
plot_settings.set_latex_settings()

In [ ]:
seed = 1
torch.manual_seed(seed)

### Define groundtruth function and observation model

In [5]:
true_func = lambda x: torch.sin(2 * torch.pi * x)
noise_std = 0.01

### Define Gaussian process

In [6]:
gp = LinearGaussianProcess(
    n_features=10000,
    nar=noise_std,
    device=torch.get_default_device(),
    dtype=torch.float64
    )
rbf_length_scale=0.1

### Define domain discretization

In [7]:
X = torch.linspace(0, 1, 1000)
X_features = random_fourier_features(X.view(-1, 1), gp.n_features, ls=rbf_length_scale, random_normals=torch.randn(1, gp.n_features//2, device=X.device))
true_function = true_func(X)

### Run Thompson sampling

In [ ]:
bo_steps = 5
X_train = []
y_train = []
for step in range(bo_steps+1):
    mean = gp.posterior_mean(X_features)
    covariance_matrix = gp.posterior_cov(X_features)
    L, Q = torch.linalg.eigh(covariance_matrix)
    sqrt_covariance_matrix = (Q * torch.sqrt(torch.maximum(L, torch.zeros_like(L)).unsqueeze(0))) @ Q.t().conj()
    posterior_sample = mean + sqrt_covariance_matrix @ torch.randn_like(mean)
    thompson_sample = torch.argmax(posterior_sample)
    if step != bo_steps:
        X_train.append(X[thompson_sample].view(()))
        y_train.append(true_function[thompson_sample] + noise_std * torch.randn(()))
        gp.add_observation(X_features[thompson_sample, :], obs=y_train[-1], perform_marginal_likelihood_maximization=True, min_obs=4)
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)

In [9]:
posterior_mean = mean
posterior_std = torch.sqrt(torch.diag(covariance_matrix))


### Plot posterior GP

In [ ]:
# plot results
fig1, (ax1) = plt.subplots(1, 1, figsize=(plot_settings.document_width/2, 2.0))
fig2, (ax2) = plt.subplots(1, 1, figsize=(plot_settings.document_width/2, 2.0))
#plt.plot(X.numpy(), true_function.numpy(), 'k:', label="True Function")
ax1.scatter(X_train.numpy(), y_train.numpy(), color='red', label="Observations", zorder=5, marker="o", s=25, edgecolors='white', linewidths=0.1)
ax1.plot(X.numpy(), posterior_mean.numpy(), 'b-', label="Posterior mean")
ax1.fill_between(
    X.flatten().numpy(),
    (posterior_mean - posterior_std).numpy(),
    (posterior_mean + posterior_std).numpy(),
    color='blue', alpha=0.2, label="Confidence interval"
)
ax1.plot(X.numpy(), posterior_sample.numpy(), 'g:', label="Sample from posterior")
ax1.scatter(X[thompson_sample:thompson_sample+1].numpy(), posterior_sample[thompson_sample:thompson_sample+1].numpy(), color='red', label="Thompson sample", zorder=5, marker='+', s=100)


# coarsely discretize
sampling_ratio = 100
X2 = X[::sampling_ratio]
posterior_mean2 = posterior_mean[::sampling_ratio]
posterior_std2 = posterior_std[::sampling_ratio]
posterior_sample2 = posterior_sample[::sampling_ratio]
thompson_sample2 = torch.argmax(posterior_sample2)

#ax2.scatter(X_train.numpy(), y_train.numpy(), color='red', label="Observations", zorder=5, marker="o")
markers, caps, bars = ax2.errorbar(
    X2.flatten().numpy(),
    posterior_mean2.numpy(),
    posterior_std2.numpy(),
    0,
    color='blue', 
    marker='s',
    markersize=10,
    elinewidth=10.0,
    alpha=0.5,
    ls='none',
    label=r"Mean \& confidence interval",
)
[bar.set_alpha(0.2) for bar in bars]
[cap.set_alpha(0.2) for cap in caps]
ax2.scatter(X2.numpy(), posterior_sample2.numpy(), color='green', label="Sample from posterior", zorder=5, marker="D", s=40, edgecolors='white', linewidths=0.1)
ax2.scatter(X2[thompson_sample2:thompson_sample2+1].numpy(), posterior_sample2[thompson_sample2:thompson_sample2+1].numpy(), color='red', label="Thompson sample", zorder=5, marker='+', s=100)


ax1.set_ylim(-1.4, 1.35)
ax2.set_ylim(-1.4, 1.35)
ax1.set_xlabel("X")
ax2.set_xlabel("X")
ax1.set_ylabel("Reward")
ax2.set_ylabel("Reward")
#ax1.legend(loc='upper right')
#ax2.legend(loc='upper right')
ax1.set_axis_off()
ax2.set_axis_off()
plt.show()

In [11]:
fig1.savefig("visualizations/results/thompson_sampling_continuous.pgf", bbox_inches="tight")

In [12]:
fig2.savefig("visualizations/results/thompson_sampling_discrete.pgf", bbox_inches="tight")

In [ ]:
cd visualizations/

In [ ]:
! bash pgf_compiler.sh thompson_sampling_continuous

In [ ]:
! bash pgf_compiler.sh thompson_sampling_discrete